# 01 — Dataset Generation (Batch)
Generate game-specific questions from PDFs using vLLM on the A100 server.

**Run on:** A100 server (after SSH tunnel is active)  
**Model:** `Qwen/Qwen2.5-7B-Instruct-AWQ` (10 GB MIG slice)

> Acknowledgement: Computational resources supported by KU Nontri AI, Kasetsart University

## Quick Start
1. Start vLLM on server: `sbatch ~/ku_prep_arena/ai/scripts/slurm_vllm.sh`
2. Open SSH tunnel (local machine): `ssh -N -L 8000:dgx-XX:8000 aip04@br1.paas.ku.ac.th`
3. Run this notebook cell by cell

In [ ]:
import os, json, sys, re
from pathlib import Path
from openai import OpenAI

# ── Connect to vLLM ──────────────────────────────────────────────────────────
BASE_URL = os.getenv("AI_BASE_URL", "http://localhost:8000/v1")
API_KEY  = os.getenv("AI_API_KEY",  "none")
MODEL    = os.getenv("AI_MODEL",    "Qwen/Qwen2.5-7B-Instruct-AWQ")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Quick health check
try:
    models = client.models.list()
    print(f"✅ Connected to: {BASE_URL}")
    print(f"✅ Available model: {models.data[0].id if models.data else 'none'}")
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("   → ตรวจสอบว่า vLLM รันอยู่และ SSH tunnel เปิดแล้ว")
    raise

## Config

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
PDF_DIR   = Path("../dataset/sample_pdfs")
OUT_DIR   = Path("../dataset/generated")
GAME_TYPES = ["flappy", "racer", "shooter", "snake", "bricks"]

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDFs   : {PDF_DIR}")
print(f"Output : {OUT_DIR}")

## Helper functions

In [ ]:
NO_CHINESE = (
    "CRITICAL: Do NOT output Chinese (中文), Japanese, Korean, Arabic, or any script "
    "not present in the document. Write ONLY in the same language as the document "
    "(Thai = ภาษาไทย, English = English)."
)

GAME_HINTS = {
    "flappy":  "Keep questions SHORT (under 12 words) and choices very brief (1-4 words). Player reads while flying.",
    "racer":   "Keep EVERY question under 12 words. Choices must be 1-4 words each — no long phrases.",
    "shooter": "Focus on IDENTIFICATION questions: 'Which term describes…?'. Include plausible distractors.",
    "snake":   "Prefer SEQUENTIAL questions: 'What is the FIRST step in…?'. Choices = different stages.",
    "bricks":  "Focus on DEFINITIONS: 'What does X mean?'. Test precise vocabulary.",
}

def extract_text_simple(pdf_path: Path, max_chars: int = 12_000) -> str:
    """Extract text from PDF using pdfplumber (or fallback to bytes)."""
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            pages = [p.extract_text() or "" for p in pdf.pages]
        return "\n".join(pages)[:max_chars]
    except ImportError:
        # Fallback: try pypdf
        try:
            from pypdf import PdfReader
            reader = PdfReader(str(pdf_path))
            text = "\n".join(p.extract_text() or "" for p in reader.pages)
            return text[:max_chars]
        except ImportError:
            raise ImportError("Install pdfplumber or pypdf: pip install pdfplumber")

def build_messages(text: str, game_type: str) -> list[dict]:
    hint = GAME_HINTS.get(game_type, "")
    system = f"{hint} {NO_CHINESE}\n\nRespond ONLY with a valid JSON array — no markdown, no explanation."
    user = f"""Create exactly 10 multiple-choice questions from the study material below.

CRITICAL RULES:
- Write ONLY in the SAME language as the document (Thai or English)
- Do NOT use Chinese, Japanese, Korean, or any other language
- Each question must have exactly 4 choices
- Only 1 correct answer per question
- Include a brief explanation

Return ONLY this JSON format:
[{{"id":1,"question":"...","choices":["A","B","C","D"],"correct":0,"explanation":"..."}}]

Study Material:
{text}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def generate_questions(text: str, game_type: str) -> list[dict]:
    messages = build_messages(text, game_type)
    res = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.5, max_tokens=2048
    )
    raw = res.choices[0].message.content or "[]"
    cleaned = re.sub(r"^```[a-z]*\n?", "", raw, flags=re.IGNORECASE)
    cleaned = re.sub(r"\n?```$", "", cleaned, flags=re.IGNORECASE).strip()
    return json.loads(cleaned)

print("✅ Functions defined")

## Generate questions for all PDFs × all game types

In [ ]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs: {[p.name for p in pdf_files]}")

results = []
for pdf_path in pdf_files:
    print(f"\n── {pdf_path.name}")
    text = extract_text(pdf_path)
    print(f"   Extracted {len(text)} chars")
    
    for game_type in tqdm(GAME_TYPES, desc="  game types"):
        out_file = OUT_DIR / f"{pdf_path.stem}_{game_type}.json"
        if out_file.exists():
            print(f"   [SKIP] {out_file.name} already exists")
            continue
        try:
            questions = call_llm(text, game_type)
            for i, q in enumerate(questions):
                q["game_type"]  = game_type
                q["source_doc"] = pdf_path.name
                q.setdefault("id", i + 1)
            out_file.write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"   ✓ {out_file.name}  ({len(questions)} questions)")
            results.append({"file": out_file.name, "n": len(questions), "ok": True})
        except Exception as e:
            print(f"   ✗ {game_type}: {e}")
            results.append({"file": f"{pdf_path.stem}_{game_type}.json", "error": str(e), "ok": False})

print(f"\nDone: {sum(r['ok'] for r in results)}/{len(results)} succeeded")